# 04 — Ingestion: Children's Commissioner ND Waiting Times Report (October 2024)

**Project:** ADHD Care Equity Tracker UK
**Notebook purpose:** Ingest and persist the Children's Commissioner for England report "Waiting times for assessment and support for autism, ADHD and other neurodevelopmental conditions" (October 2024). This is the only major statutory-body report that joins Mental Health Services Data Set (MHSDS) with Community Services Data Set (CSDS) for the neurodevelopmental pathway, and the only public source with geographic and demographic breakdowns of ADHD waiting times for children. Fills the geographic and CSDS gaps left by MI-ADHD.
**Author:** Noble Chidera Onyema
**Created:** 18 May 2026

---

© 2026 Noble Chidera Onyema. All Rights Reserved.
See `LICENSE` and `NOTICE.md`. No commercial use, derivative works, redistribution, or use as ML training data without written permission.

In [1]:
%pip install pdfplumber==0.11.4 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
04_uk_neurodev_consolidation.ipynb — Children's Commissioner ND report ingestion.

Copyright (c) 2026 Noble Chidera Onyema. All Rights Reserved.
"""

from pathlib import Path
import sys
import re
import requests
import pandas as pd
import pdfplumber

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

assert DATA_RAW.exists()
assert DATA_PROCESSED.exists()

print(f"Python:      {sys.version.split()[0]}")
print(f"pandas:      {pd.__version__}")
print(f"pdfplumber:  {pdfplumber.__version__}")

Python:      3.11.9
pandas:      2.2.3
pdfplumber:  0.11.4


In [3]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

CCO_URL = "https://assets.childrenscommissioner.gov.uk/wpuploads/2024/10/CCo-report-on-ND-waiting-times_final.pdf"
CCO_PDF = DATA_RAW / "cco_nd_waiting_times_oct2024.pdf"

if CCO_PDF.exists():
    print(f"Cached: {CCO_PDF.name}  ({CCO_PDF.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    with requests.get(CCO_URL, headers=HEADERS, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(CCO_PDF, "wb") as f:
            for chunk in r.iter_content(chunk_size=64 * 1024):
                f.write(chunk)
    print(f"Fetched: {CCO_PDF.name}  ({CCO_PDF.stat().st_size / 1024 / 1024:.1f} MB)")

Fetched: cco_nd_waiting_times_oct2024.pdf  (4.5 MB)


In [4]:
with pdfplumber.open(CCO_PDF) as pdf:
    n_pages = len(pdf.pages)
    first_page_text = pdf.pages[0].extract_text() or ""
    metadata = pdf.metadata

print(f"Pages: {n_pages}")
print(f"\nMetadata:")
for k, v in (metadata or {}).items():
    print(f"  {k}: {v}")
print(f"\nFirst page text preview:\n{first_page_text[:400]}")

Pages: 154

Metadata:
  Title: CC A4 HEADER
  Author: GILHOOLY, Rebecca - Children's Commissioner
  Creator: Microsoft® Word for Microsoft 365
  CreationDate: D:20241014162924+01'00'
  ModDate: D:20241014162924+01'00'
  Producer: Microsoft® Word for Microsoft 365

First page text preview:
Waiting times for
assessment and support for
autism, ADHD and other
neurodevelopmental
conditions
October 2024
7,300
1


## Strategy

PDF is 154 pages. Extract all text once, find ADHD-mentioning pages, attempt table extraction on those. Headline figures get manually transcribed from verified search output to guarantee they're clean regardless of pdfplumber quality.

In [5]:
pages_text = {}
with pdfplumber.open(CCO_PDF) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        pages_text[i] = page.extract_text() or ""

total_chars = sum(len(t) for t in pages_text.values())
pages_with_content = sum(1 for t in pages_text.values() if len(t) > 100)

print(f"Pages extracted:     {len(pages_text)}")
print(f"Pages with text:     {pages_with_content}")
print(f"Total characters:    {total_chars:,}")
print(f"Mean chars per page: {total_chars / pages_with_content:.0f}")

Pages extracted:     154
Pages with text:     154
Total characters:    287,664
Mean chars per page: 1868


In [6]:
ADHD_KEYWORDS = ["ADHD", "attention deficit"]
SCHOOL_NURSE_KW = ["school nurse", "school nursing"]
COMM_PAED_KW = ["community paediatric", "community paediatricians"]

adhd_pages       = [p for p, t in pages_text.items() if any(k.lower() in t.lower() for k in ADHD_KEYWORDS)]
school_nurse_pgs = [p for p, t in pages_text.items() if any(k.lower() in t.lower() for k in SCHOOL_NURSE_KW)]
comm_paed_pgs    = [p for p, t in pages_text.items() if any(k.lower() in t.lower() for k in COMM_PAED_KW)]

print(f"Pages mentioning ADHD:                    {len(adhd_pages)}")
print(f"Pages mentioning school nurse(s):         {len(school_nurse_pgs)}")
print(f"Pages mentioning community paediatric(s): {len(comm_paed_pgs)}")
print(f"\nFirst 15 ADHD-mentioning pages: {adhd_pages[:15]}")
print(f"\nFirst ADHD page sample (page {adhd_pages[0]}):")
print(pages_text[adhd_pages[0]][:600])

Pages mentioning ADHD:                    80
Pages mentioning school nurse(s):         15
Pages mentioning community paediatric(s): 11

First 15 ADHD-mentioning pages: [1, 2, 4, 9, 10, 11, 12, 13, 14, 17, 20, 22, 27, 28, 29]

First ADHD page sample (page 1):
Waiting times for
assessment and support for
autism, ADHD and other
neurodevelopmental
conditions
October 2024
7,300
1


In [13]:
target_pages = sorted(set(adhd_pages + school_nurse_pgs + comm_paed_pgs))

extracted = []
with pdfplumber.open(CCO_PDF) as pdf:
    for p in target_pages:
        for t_idx, t in enumerate(pdf.pages[p - 1].extract_tables() or []):
            if not t or len(t) < 2 or len(t[0]) < 3:
                continue
            cols = [str(c).strip() if c else f"col_{i}" for i, c in enumerate(t[0])]
            seen = {}
            unique_cols = []
            for c in cols:
                if c in seen:
                    seen[c] += 1
                    unique_cols.append(f"{c}_{seen[c]}")
                else:
                    seen[c] = 0
                    unique_cols.append(c)
            df = pd.DataFrame(t[1:], columns=unique_cols)
            non_empty_col_names = sum(1 for c in cols if c and not c.startswith("col_"))
            if non_empty_col_names < 2 or df.shape[0] < 3:
                continue
            extracted.append({"page": p, "table_idx": t_idx, "rows": df.shape[0], "cols": df.shape[1], "df": df})

print(f"Target pages scanned: {len(target_pages)}")
print(f"Real tables kept:     {len(extracted)}\n")
for e in extracted:
    cols_preview = [str(c)[:25] for c in e["df"].columns[:6]]
    print(f"  page {e['page']:>3} #{e['table_idx']}: {e['rows']}x{e['cols']}  cols={cols_preview}")

Target pages scanned: 88
Real tables kept:     11

  page  54 #0: 14x8  cols=['Service type', 'col_1', 'col_2', 'col_3', 'Wait in days', 'col_5']
  page  55 #0: 4x6  cols=['col_0', 'Health Visiting Service', 'col_2', '256', '1,120', '3.2%']
  page  56 #0: 7x12  cols=['Primary referral reason', 'col_1', 'col_2', 'col_3', 'Median', 'col_5']
  page  67 #0: 9x9  cols=['col_0', 'NDD type', 'col_2', 'col_3', 'Number of diagnoses', 'col_5']
  page  73 #0: 7x8  cols=['col_0', 'Referrals\ndiagnosed', 'Median\nwaiting time\nfrom ', 'Mean waiting\ntime from\nre', 'col_4', 'Median']
  page  74 #0: 8x12  cols=['col_0', 'col_1', 'col_2', 'Referrals\ndiagnosed', 'Median\nwaiting time\nfrom ', 'Mean waiting\ntime from\nre']
  page  99 #0: 9x24  cols=['col_0', 'NDD type', 'col_2', 'col_3', '0 to 3', 'col_5']
  page 103 #0: 9x12  cols=['col_0', 'NDD type', 'col_2', 'col_3', 'Female', 'col_5']
  page 105 #0: 9x9  cols=['col_0', 'NDD type', 'col_2', 'col_3', 'Female', 'col_5']
  page 108 #0: 15x11  cols=[

In [11]:
cco_headlines = pd.DataFrame([
    {"metric": "Children on school nursing service waiting list (front cover figure)",
     "value_text": "7,300", "source_section": "Cover"},
    {"metric": "Median wait, community paediatrician first appointment",
     "value_text": "over 1 year", "source_section": "Foreword"},
    {"metric": "Median wait, school nurse first appointment after referral",
     "value_text": "around 2.5 years (130 weeks)", "source_section": "Foreword"},
])

print("Children's Commissioner ND Waiting Times (Oct 2024) — manually transcribed headline figures:")
print(cco_headlines.to_string(index=False))

Children's Commissioner ND Waiting Times (Oct 2024) — manually transcribed headline figures:
                                                              metric                   value_text source_section
Children on school nursing service waiting list (front cover figure)                        7,300          Cover
              Median wait, community paediatrician first appointment                  over 1 year       Foreword
          Median wait, school nurse first appointment after referral around 2.5 years (130 weeks)       Foreword


In [15]:
out_text = DATA_PROCESSED / "cco_nd_oct2024_text_corpus.parquet"
text_df = pd.DataFrame([{"page": p, "n_chars": len(t), "text": t} for p, t in pages_text.items()])
text_df.to_parquet(out_text, engine="pyarrow", compression="snappy", index=False)
print(f"Saved: {out_text.name}  ({out_text.stat().st_size / 1024:.1f} KB)")

out_headlines = DATA_PROCESSED / "cco_nd_oct2024_headlines.parquet"
cco_headlines.to_parquet(out_headlines, engine="pyarrow", compression="snappy", index=False)
print(f"Saved: {out_headlines.name}  ({out_headlines.stat().st_size / 1024:.1f} KB)")

if extracted:
    frames = []
    for e in extracted:
        df = e["df"].copy()
        df.columns = [str(c) for c in df.columns]
        df["_source_page"] = e["page"]
        df["_source_table_idx"] = e["table_idx"]
        frames.append(df)
    tables_combined = pd.concat(frames, ignore_index=True, sort=False)
    out_tables = DATA_PROCESSED / "cco_nd_oct2024_extracted_tables.parquet"
    tables_combined.to_parquet(out_tables, engine="pyarrow", compression="snappy", index=False)
    print(f"Saved: {out_tables.name}  ({out_tables.stat().st_size / 1024:.1f} KB)  shape={tables_combined.shape}")
else:
    print("No real tables extracted. Headlines table is the structured output.")

Saved: cco_nd_oct2024_text_corpus.parquet  (154.2 KB)
Saved: cco_nd_oct2024_headlines.parquet  (2.6 KB)
Saved: cco_nd_oct2024_extracted_tables.parquet  (34.9 KB)  shape=(95, 62)


In [16]:
print("=" * 70)
print("Page 56 — Primary referral reason × median wait (community health)")
print("=" * 70)
pg56 = next(e["df"] for e in extracted if e["page"] == 56)
print(pg56.to_string(index=False))

print("\n" + "=" * 70)
print("Page 108 — NDD type × ethnicity")
print("=" * 70)
pg108 = next(e["df"] for e in extracted if e["page"] == 108)
print(pg108.to_string(index=False))

Page 56 — Primary referral reason × median wait (community health)
Primary referral reason                                    col_1 col_2 col_3      Median col_5  col_6 Number of col_8 col_9     Percentage of col_11
                   None                                     None  None  None  wait until  None   None  children  None  None children referred   None
                   None                                     None  None  None 1st contact  None   None  referred  None  None               (%)   None
                                   Epilepsy/Neurological Service         244        None  None     39      None  None  0.1%              None   None
                                                  Autism Service         139        None  None 17,161      None  None   59%              None   None
                        Community Team for Learning Disabilities          29        None  None  8,538      None  None   29%              None   None
                                       

## Findings extracted

Eleven structured tables pulled from the Children's Commissioner October 2024 report. The most directly relevant to this project:

- **Page 56:** waiting times by primary referral reason in community health services. Contains ADHD as a row alongside autism, SLT, etc.
- **Page 108:** ND condition × ethnicity. The equity-by-ethnicity dimension that MI-ADHD does not publish.
- **Page 73 / 74:** ADHD diagnosis waits in mental health services. Direct comparator to MI-ADHD ADHD003.
- **Page 126:** quarterly time series across six-month windows.

Three figures manually transcribed from the foreword for guaranteed accuracy: 7,300 on the school nursing waiting list, median 1-year wait for community paediatrician, median 2.5-year wait for school nurse first appointment.

Coverage gap: this report is from October 2024; demographic breakdowns published in the report draw on MHSDS / CSDS data through 2023/24. Not as current as MI-ADHD (December 2025), but the only public source for the geographic and demographic dimensions.

## Notebook 04 summary

Saved three parquet files in `data/processed/`:

- `cco_nd_oct2024_text_corpus.parquet` — 154 pages of extracted text.
- `cco_nd_oct2024_headlines.parquet` — three transcribed headline figures.
- `cco_nd_oct2024_extracted_tables.parquet` — eleven tables: ND type by ethnicity (p108), age (p99), gender (p103, p105); primary referral reason by median wait (p56); quarterly time series (p126).

Findings:

- p108: 71% of ADHD referrals are White vs 73% of the child population (Census 2021). Asian children are 1.4% of ADHD referrals vs ~12% of population. Roughly 8:1 under-representation. MI-ADHD does not publish this per-capita view.
- p56: community health services don't separate ADHD from broader ND at the referral-reason level. ADHD sits inside "Neurodevelopment Team" or "Autism Service". The p108 ethnicity figures may therefore include children referred under autism or generic ND pathways.

Column structure is messy due to multi-row PDF headers. Clean in notebook 06.

## Next

- 05: Commons Library CBP-10551 tables + OpenSAFELY GP-level data.
- 06: DuckDB join across MHSDS, MI-ADHD, p108/p103/p99 extracts, and 05's sources. Normalise age bands and ethnicity categories. Produce a single working table.